In [2]:
import os
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
load_dotenv()

api_key = os.getenv("OPENROUTER_API_KEY")
if not api_key:
    raise ValueError("Не найден OPENROUTER_API_KEY. Установи переменную окружения.")

llm = ChatOpenAI(
    model="qwen3.8-27b",                      
    api_key=api_key,
    base_url="https://openrouter.ai/api/v1",
    temperature=0,                                # для стабильного JSON в части 2
    extra_body={"reasoning": {"enabled": False}}, # выключаем размышление 
)

# Проверяем, что соединение работает
response = llm.invoke("Скажи одно слово: работает?")
print("Тип ответа:", type(response))
print("Содержимое:", response.content)
print("usage_metadata:", response.usage_metadata)

Тип ответа: <class 'langchain_core.messages.ai.AIMessage'>
Содержимое: Да
usage_metadata: {'input_tokens': 20, 'output_tokens': 2, 'total_tokens': 22, 'input_token_details': {'audio': 0, 'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


## Часть 1. Базовая цепочка LangChain

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Параметризованный промпт:
#    - system: роль модели
#    - human: переменная query (запрос) и language (язык ответа)
prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Ты — ассистент интернет-магазина. Твоя задача — определить категорию товара "
     "по запросу пользователя и кратко ответить. Отвечай на языке: {language}."),
    ("human", "{query}"),
])

# LCEL-цепочка: промпт → LLM → парсер строки
chain = prompt | llm | StrOutputParser()

Сравнение: прямой вызов LLM vs вызов цепочки


In [ ]:
direct_response = llm.invoke("Определи категорию товара: ноутбук до 70 тысяч рублей")
chain_response = chain.invoke({
    "query": "Определи категорию товара: ноутбук до 70 тысяч рублей",
    "language": "русский",
})

print("Прямой вызов LLM:")
print("  тип:", type(direct_response))
print("  содержимое:", direct_response.content[:100], "...")
print()
print("Вызов цепочки:")
print("  тип:", type(chain_response))
print("  содержимое:", chain_response[:100], "...")

Прямой вызов LLM:
  тип: <class 'langchain_core.messages.ai.AIMessage'>
  содержимое: Категория товара: **Ноутбуки** (или **Компьютеры и ноутбуки**). ...

Вызов цепочки:
  тип: <class 'langchain_core.messages.base.TextAccessor'>
  содержимое: Ноутбуки ...


In [9]:
direct_response

AIMessage(content='Категория товара: **Ноутбуки** (или **Компьютеры и ноутбуки**).', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 29, 'total_tokens': 52, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'text_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None, 'video_tokens': 0}, 'cost': 8.665e-05, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 8.665e-05, 'upstream_inference_prompt_cost': 1.305e-05, 'upstream_inference_completions_cost': 7.36e-05}}, 'model_provider': 'openai', 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': None, 'id': 'gen-1794333158-L0sWwiKBWSNa8wRspbUY', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a1c697-2cd7-76a6-b94e-926afc92f238-0', tool_calls=[], inva

In [ ]:
print(type(chain_response).__mro__)   # посмотреть цепочку классов
print(isinstance(chain_response, str))  

(<class 'langchain_core.messages.base.TextAccessor'>, <class 'str'>, <class 'object'>)
True


In [5]:
queries = [
    {"query": "Найди ноутбуки дешевле 70 000 рублей", "language": "русский"},
    {"query": "Покажи самые дорогие смартфоны", "language": "русский"},
    {"query": "Хочу посмотреть планшеты", "language": "русский"},
]

batch_results = chain.batch(queries)

for q, r in zip(queries, batch_results):
    print(f"Запрос: {q['query']}")
    print(f"Ответ:  {r}\n")

Запрос: Найди ноутбуки дешевле 70 000 рублей
Ответ:  Ноутбуки

Запрос: Покажи самые дорогие смартфоны
Ответ:  Смартфоны

Запрос: Хочу посмотреть планшеты
Ответ:  Планшеты



## Часть 2. Structured Output и Pydantic

In [13]:
from typing import Literal, Optional
from pydantic import BaseModel, Field

class ProductQuery(BaseModel):
    category: Literal["laptop", "smartphone", "tablet", "other"] = Field(
        description="Категория товара. Если категория не входит в список поддерживаемых, используй other."
    )
    max_price: Optional[int] = Field(
        default=None,
        ge=0,
        description="Верхняя граница цены в рублях. Если ограничения нет — null."
    )
    sort_by: Literal["price_asc", "price_desc"] = Field(
        default="price_desc",
        description="Направление сортировки по цене: price_asc — по возрастанию, price_desc — по убыванию."
    )

In [14]:
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import ChatPromptTemplate

parser = PydanticOutputParser(pydantic_object=ProductQuery)
format_instructions = parser.get_format_instructions()

prompt_structured = ChatPromptTemplate.from_messages([
    ("system",
     "Ты — ассистент интернет-магазина. Преобразуй запрос пользователя в структурированные "
     "параметры поиска товаров.\n\n{format_instructions}"),
    ("human", "{query}"),
]).partial(format_instructions=format_instructions)

chain_structured = prompt_structured | llm | parser

In [15]:
result = chain_structured.invoke({"query": "Найди ноутбуки дешевле 70 000 рублей"})
print("Тип результата:", type(result))
print("Объект:", result)
print("JSON:", result.model_dump_json(indent=2))

Тип результата: <class '__main__.ProductQuery'>
Объект: category='laptop' max_price=70000 sort_by='price_asc'
JSON: {
  "category": "laptop",
  "max_price": 70000,
  "sort_by": "price_asc"
}


In [16]:
test_queries = [
    "Найди ноутбуки дешевле 70 000 рублей",
    "Покажи самые дорогие смартфоны",
    "Хочу посмотреть планшеты",
    "Покажи ноутбуки до 50 тысяч",
    "Найди холодильники",
]

for i, q in enumerate(test_queries, 1):
    print(f"Запрос {i}: {q}")
    try:
        result = chain_structured.invoke({"query": q})
        print(f"JSON: {result.model_dump_json(indent=2)}")
        print(f"Проверка полей:")
        print(f"  category  = {result.category!r} (допустимо: laptop/smartphone/tablet/other)")
        print(f"  max_price = {result.max_price!r} (None или неотрицательное целое)")
        print(f"  sort_by   = {result.sort_by!r} (price_asc / price_desc)")
    except Exception as e:
        print(f"ОШИБКА: {type(e).__name__}: {e}")

Запрос 1: Найди ноутбуки дешевле 70 000 рублей
JSON: {
  "category": "laptop",
  "max_price": 70000,
  "sort_by": "price_asc"
}
Проверка полей:
  category  = 'laptop' (допустимо: laptop/smartphone/tablet/other)
  max_price = 70000 (None или неотрицательное целое)
  sort_by   = 'price_asc' (price_asc / price_desc)
Запрос 2: Покажи самые дорогие смартфоны
JSON: {
  "category": "smartphone",
  "max_price": null,
  "sort_by": "price_desc"
}
Проверка полей:
  category  = 'smartphone' (допустимо: laptop/smartphone/tablet/other)
  max_price = None (None или неотрицательное целое)
  sort_by   = 'price_desc' (price_asc / price_desc)
Запрос 3: Хочу посмотреть планшеты
JSON: {
  "category": "tablet",
  "max_price": null,
  "sort_by": "price_desc"
}
Проверка полей:
  category  = 'tablet' (допустимо: laptop/smartphone/tablet/other)
  max_price = None (None или неотрицательное целое)
  sort_by   = 'price_desc' (price_asc / price_desc)
Запрос 4: Покажи ноутбуки до 50 тысяч
JSON: {
  "category": "lapt

In [17]:
ambiguous_queries = [
    "Что-нибудь недорогое для работы, но не телефон",
    "Покажи топовые варианты, но чтобы не слишком дорого",
]

for q in ambiguous_queries:
    print(f"Неоднозначный запрос: {q}")
    try:
        result = chain_structured.invoke({"query": q})
        print(f"JSON: {result.model_dump_json(indent=2)}")
    except Exception as e:
        print(f"ОШИБКА: {type(e).__name__}: {e}")

Неоднозначный запрос: Что-нибудь недорогое для работы, но не телефон
JSON: {
  "category": "laptop",
  "max_price": 50000,
  "sort_by": "price_asc"
}
Неоднозначный запрос: Покажи топовые варианты, но чтобы не слишком дорого
JSON: {
  "category": "other",
  "max_price": null,
  "sort_by": "price_desc"
}


 во втором вопросе - спрашивает "не слишком дорого" - а максимальной цены не поставила модель, зотя в первом случае ограничила 50000

модель не смогла извлечь категорию и числовое ограничение из формулировки без явных якорей: category="other", max_price=None. 

при отсутствии явных сущностей модель либо угадывает (как в первом запросе), либо откатывается к значениям по умолчанию. В реальной системе такие запросы следует распознавать и задавать пользователю уточняющий вопрос, а не выполнять поиск с other

In [18]:
try:
    result = chain_structured.invoke({"query": 'Что-нибудь до 30 тысяч'})
    print(f"JSON: {result.model_dump_json(indent=2)}")
except Exception as e:
    print(f"ОШИБКА: {type(e).__name__}: {e}")

JSON: {
  "category": "other",
  "max_price": 30000,
  "sort_by": "price_desc"
}


 число извлекла чётко, а категорию не угадывала, поставила other, потому что в тексте нет никаких указаний на тип товара

In [ ]:
import json, os

os.makedirs("results", exist_ok=True)

test_queries = [
    "Найди ноутбуки дешевле 70 000 рублей",
    "Покажи самые дорогие смартфоны",
    "Хочу посмотреть планшеты",
    "Покажи ноутбуки до 50 тысяч",
    "Найди холодильники",
    "Что-нибудь недорогое для работы, но не телефон",
    "Покажи топовые варианты, но чтобы не слишком дорого",
    "Что-нибудь до 30 тысяч",
]

records = []
for q in test_queries:
    try:
        result = chain_structured.invoke({"query": q})
        records.append({
            "query": q,
            "json": json.loads(result.model_dump_json()),
            "valid": True,
            "error": None,
        })
        print(f"OK: {q} -> {result.model_dump_json()}")
    except Exception as e:
        records.append({
            "query": q,
            "json": None,
            "valid": False,
            "error": f"{type(e).__name__}: {e}",
        })
        print(f"FAIL: {q} -> {e}")

with open("results/part2_3_structured_tests.json", "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=2)

print(f"\nСохранено {len(records)} записей в results/part2_3_structured_tests.json")

**Задание 2.4 — сравнение режима размышления**

Проверка: включается ли reasoning у текущей модели


In [19]:
llm_thinking = ChatOpenAI(
    model="qwen3.8-27b",   
    api_key=api_key,
    base_url="https://openrouter.ai/api/v1",
    temperature=0,
    extra_body={"reasoning": {"enabled": True}},
)

resp = llm_thinking.invoke("Сколько будет 17 * 23? Ответь одним числом.")
print("content:", repr(resp.content))
print("usage_metadata:", resp.usage_metadata)
print("response_metadata:", resp.response_metadata)

content: '391'
usage_metadata: {'input_tokens': 70, 'output_tokens': 58, 'total_tokens': 128, 'input_token_details': {'audio': 0, 'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'audio': 0, 'reasoning': 54}}
response_metadata: {'token_usage': {'completion_tokens': 58, 'prompt_tokens': 70, 'total_tokens': 128, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 54, 'rejected_prediction_tokens': None, 'text_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None, 'video_tokens': 0}, 'cost': 0.000159, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0.000159, 'upstream_inference_prompt_cost': 1.4e-05, 'upstream_inference_completions_cost': 0.000145}}, 'model_provider': 'openai', 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': None, 'id': 'gen-1794335873-XcPcDyEdN26AOVmrzr4C', 'finish_reason'

In [20]:
import time

llm_off = ChatOpenAI(
    model="qwen3.8-27b", api_key=api_key, base_url="https://openrouter.ai/api/v1",
    temperature=0, extra_body={"reasoning": {"enabled": False}},
)
llm_on = ChatOpenAI(
    model="qwen3.8-27b", api_key=api_key, base_url="https://openrouter.ai/api/v1",
    temperature=0, extra_body={"reasoning": {"enabled": True}},
)

chain_off = prompt_structured | llm_off | parser
chain_on  = prompt_structured | llm_on  | parser

ambiguous = [
    "Что-нибудь недорогое для работы, но не телефон",
    "Покажи топовые варианты, но чтобы не слишком дорого",
]

for q in ambiguous:
    print(f"\n{'='*60}\nЗапрос: {q}")
    for name, chain in [("OFF", chain_off), ("ON", chain_on)]:
        t0 = time.time()
        try:
            result = chain.invoke({"query": q})
            elapsed = time.time() - t0
            print(f"\n[{name}] {elapsed:.2f} s")
            print(f"  JSON: {result.model_dump_json()}")
        except Exception as e:
            elapsed = time.time() - t0
            print(f"\n[{name}] {elapsed:.2f} s — ОШИБКА: {type(e).__name__}: {str(e)[:200]}")


Запрос: Что-нибудь недорогое для работы, но не телефон

[OFF] 2.74 s
  JSON: {"category":"laptop","max_price":50000,"sort_by":"price_asc"}

[ON] 0.76 s
  JSON: {"category":"laptop","max_price":null,"sort_by":"price_asc"}

Запрос: Покажи топовые варианты, но чтобы не слишком дорого

[OFF] 1.30 s
  JSON: {"category":"other","max_price":null,"sort_by":"price_desc"}

[ON] 4.05 s
  JSON: {"category":"other","max_price":null,"sort_by":"price_asc"}


посмотрим сырой content и токены

In [21]:
messages = prompt_structured.format_messages(query="Покажи топовые варианты, но чтобы не слишком дорого")
for name, llm_x in [("OFF", llm_off), ("ON", llm_on)]:
    t0 = time.time()
    resp = llm_x.invoke(messages)
    elapsed = time.time() - t0
    print(f"\n[{name}] {elapsed:.2f} s")
    print(f"  content: {resp.content!r}")
    print(f"  usage: {resp.usage_metadata}")


[OFF] 2.29 s
  content: '```json\n{\n  "category": "other",\n  "max_price": null,\n  "sort_by": "price_desc"\n}\n```'
  usage: {'input_tokens': 364, 'output_tokens': 35, 'total_tokens': 399, 'input_token_details': {'audio': 0, 'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}

[ON] 4.87 s
  content: '\n\n{\n  "category": "other",\n  "max_price": null,\n  "sort_by": "price_asc"\n}'
  usage: {'input_tokens': 400, 'output_tokens': 394, 'total_tokens': 794, 'input_token_details': {'audio': 0, 'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'audio': 0, 'reasoning': 363}}


In [ ]:
import time

comparison = []
ambiguous = [
    "Что-нибудь недорогое для работы, но не телефон",
    "Покажи топовые варианты, но чтобы не слишком дорого",
]

for q in ambiguous:
    messages = prompt_structured.format_messages(query=q)
    entry = {"query": q, "off": {}, "on": {}}
    for name, llm_x, chain_x in [("off", llm_off, chain_off), ("on", llm_on, chain_on)]:
        # сырой ответ LLM — для токенов и content
        t0 = time.time()
        raw = llm_x.invoke(messages)
        raw_time = time.time() - t0
        # вызов через цепочку — для проверки, что парсер справляется
        t0 = time.time()
        try:
            parsed = chain_x.invoke({"query": q})
            chain_time = time.time() - t0
            parsed_json = json.loads(parsed.model_dump_json())
            chain_error = None
        except Exception as e:
            chain_time = time.time() - t0
            parsed_json = None
            chain_error = f"{type(e).__name__}: {e}"
        entry[name] = {
            "raw_content": raw.content,
            "raw_time_sec": round(raw_time, 2),
            "chain_time_sec": round(chain_time, 2),
            "usage_metadata": raw.usage_metadata,
            "parsed_json": parsed_json,
            "chain_error": chain_error,
        }
    comparison.append(entry)

with open("results/part2_4_reasoning_comparison.json", "w", encoding="utf-8") as f:
    json.dump(comparison, f, ensure_ascii=False, indent=2)

print("Сохранено в results/part2_4_reasoning_comparison.json")